In [ ]:
from pyspark.sql  import SparkSession
import pandas as pd
import requests
import json
from google.cloud import storage
import datetime

#initialize spark session
spark = SparkSession.builder.appName("customerreviewAPI").getOrCreate()

#API End point
api_url = "https://6a5dc35c0ad09982aef76ae0.mockapi.io/retailer/reviews"

#fetch data from api response
response = requests.get(api_url)

if response.status_code == 200:
    data = response.json()
    print(f"Successfully fetched {len(data)} records")
else:
    print(f"Failed to fetch data.Status code:{response.status_code}")
    exit()
#convert api data to pandas
df_pandas  = pd.DataFrame(data)

#get data fo file name
today = datetime.datetime.today().strftime('%Y%m%d')

#defining file path
local_parquet_file = f"/tmp/customer_reviews_{today}.parquet"
gcs_bucket = "retailer-datalake-project-270326"
gcs_path = f"landing/customer_reviews/customer_reviews_{today}.parquet"

#save pandas dataframe to local file
df_pandas.to_parquet(local_parquet_file,index= False)

#upload paquet file to gcs
storage_client = storage.Client()
bucket = storage_client.bucket(gcs_bucket)
blob = bucket.blob(gcs_path)
blob.upload_from_filename(local_parquet_file)

print(f"✅ Data successfully written to gs://{gcs_bucket}/{gcs_path}")